In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

rng = np.random.default_rng(42)
n = 5000

X = pd.DataFrame({
    'amount': rng.exponential(100, n),
    'velocity' : rng.normal(0, 1, n),
    'hour' : rng.integers(0, 24, n),
    'v1' : rng.normal(0, 2, n),
    'v2' : rng.normal(0, 2, n),
})

X.loc[rng.choice(n, int(n*0.20), replace=False), 'amount'] = np.nan
X.loc[rng.choice(n, int(n*0.40), replace=False), 'v2'] = np.nan

y = pd.Series((rng.random(n) < 0.035).astype(int))

print("--Before Imputation--")
print(X.isnull().sum())
print(f"Total NaN cells: {X.isnull().sum().sum()}")

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

print("\n-- After Imputation --")
print(pd.DataFrame(X_imputed).isnull().sum())
print("NaN count could be zero above")


print(f"\nMedian values used to fill NaN:")
for col, med in zip(X.columns, imputer.statistics_):
    print(f"{col:12s}: {med:.4f}")

--Before Imputation--
amount      1000
velocity       0
hour           0
v1             0
v2          2000
dtype: int64
Total NaN cells: 3000

-- After Imputation --
0    0
1    0
2    0
3    0
4    0
dtype: int64
NaN count could be zero above

Median values used to fill NaN:
amount      : 69.2326
velocity    : -0.0197
hour        : 12.0000
v1          : -0.0040
v2          : 0.1056


In [4]:
print("-- Before scaling --")
print(pd.DataFrame(X_imputed, columns=X.columns).describe().loc[['mean','std']])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("-- After scaling --")
print(pd.DataFrame(X_scaled, columns=X.columns).describe().loc[['mean','std']].round(4))


-- Before scaling --
         amount  velocity     hour        v1        v2
mean  92.315531  0.000856  11.6102  0.045429  0.084099
std   88.606712  1.013096   6.8828  2.012936  1.530745
-- After scaling --
      amount  velocity    hour      v1      v2
mean  0.0000   -0.0000 -0.0000 -0.0000 -0.0000
std   1.0001    1.0001  1.0001  1.0001  1.0001


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe_no_weight = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(max_iter=500, random_state=42))
])
pipe_no_weight.fit(X_train, y_train)
prob_no = pipe_no_weight.predict_proba(X_test)[:, 1]
pred_no = (prob_no >= 0.5).astype(int)

pipe_balanced = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(
        class_weight='balanced', max_iter=500, random_state=42
    ))
])
pipe_balanced.fit(X_train, y_train)
prob_bal = pipe_balanced.predict_proba(X_test)[:, 1]
pred_bal = (prob_bal >= 0.5).astype(int)

print("---WITHOUT class_weight--")
print(f"  Accuracy : {accuracy_score(y_test, pred_no)*100:.2f}%")
print(f"  Recall   : {recall_score(y_test, pred_no)*100:.2f}%")
print(f"  AUC      : {roc_auc_score(y_test, prob_no):.4f}")

print("\n-- WITH class_weight='balanced' --")
print(f"  Accuracy : {accuracy_score(y_test, pred_bal)*100:.2f}%")
print(f"  Recall   : {recall_score(y_test, pred_bal)*100:.2f}%")
print(f"  AUC      : {roc_auc_score(y_test, prob_bal):.4f}")

print("\nAccuracy went DOWN. Recall went UP. This is correct behaviour.")

---WITHOUT class_weight--
  Accuracy : 96.30%
  Recall   : 0.00%
  AUC      : 0.5119

-- WITH class_weight='balanced' --
  Accuracy : 54.20%
  Recall   : 43.24%
  AUC      : 0.5070

Accuracy went DOWN. Recall went UP. This is correct behaviour.


In [6]:
probs = pipe_balanced.predict_proba(X_test)

print("Shape of predict_proba output:", probs.shape)
print("\nFirst 10 rows:")
print(pd.DataFrame(probs, columns=['P(legit)', 'P(fraud)']).head(10).round(4))
print("\nDo they sum to 1.0?")
print((probs.sum(axis=1).round(4) == 1.0).all())


fraud_probs = probs[:, 1]
print(f"\nFraud probability stats:")
print(f"  Min  : {fraud_probs.min():.4f}")
print(f"  Max  : {fraud_probs.max():.4f}")
print(f"  Mean : {fraud_probs.mean():.4f}")

threshold = 0.5
y_pred = (fraud_probs >= threshold).astype(int)
print(f"\nAt threshold={threshold}: {y_pred.sum()} transactions flagged as fraud")

Shape of predict_proba output: (1000, 2)

First 10 rows:
   P(legit)  P(fraud)
0    0.5562    0.4438
1    0.4854    0.5146
2    0.4867    0.5133
3    0.4516    0.5484
4    0.4778    0.5222
5    0.4791    0.5209
6    0.5557    0.4443
7    0.5171    0.4829
8    0.5134    0.4866
9    0.5628    0.4372

Do they sum to 1.0?
True

Fraud probability stats:
  Min  : 0.3807
  Max  : 0.6070
  Mean : 0.4948

At threshold=0.5: 453 transactions flagged as fraud


In [7]:
import warnings

# What happens with too few iterations?
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    
    pipe_lowiter = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('scaler',     StandardScaler()),
        ('classifier', LogisticRegression(max_iter=5, random_state=42))
    ])
    pipe_lowiter.fit(X_train, y_train)
    
    if w:
        print("WARNING CAUGHT:")
        print(str(w[0].message))
        print("\nThis means the model stopped before finding the best weights.")
        print("It's like stopping a calculation halfway through.")
    else:
        print("No warning — converged even in 5 iterations (simple data)")

WARNING CAUGHT:
lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

This means the model stopped before finding the best weights.
It's like stopping a calculation halfway through.
